In [ ]:
import os

# Trellis 1 - Multiview/Gaussians

In [ ]:
import os
import glob
import shutil
import tempfile
from PIL import Image
from gradio_client import Client, handle_file

# Paths
SAMPLE_DIR = "/home/ubuntu/workspace/open_alpasim/data/open-gen-sensorsim-artifacts/assets/nvidia_dataset_artifacts/sample/multi_view_imgs"
OUTPUT_DIR = "/home/ubuntu/workspace/open_alpasim/data/open-gen-sensorsim-artifacts/assets/nvidia_dataset_artifacts/sample/assets"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TIMEOUT = 300
MAX_MULTIVIEW_IMAGES = 4  # TRELLIS works best with 2-4 views; cap to avoid OOM

client = Client("microsoft/TRELLIS")


# -------------------------
# Image preparation
# -------------------------
def prepare_image(path: str) -> str:
    """Resize and save image to a temp PNG, return the temp path."""
    img = Image.open(path).convert("RGB")
    img = img.resize((512, 512))
    tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    img.save(tmp.name)
    return tmp.name


# -------------------------
# Main pipeline
# -------------------------
def run_trellis_pipeline(image_folder: str, multiview: bool = True):
    image_paths = sorted(glob.glob(os.path.join(image_folder, "*.png")))
    if not image_paths:
        raise FileNotFoundError(f"No PNG files found in {image_folder}")

    print(f"Found {len(image_paths)} images — multiview={multiview}")

    # -------------------------
    # STEP 0: Start session
    # -------------------------
    print("Starting session...")
    client.predict(api_name="/start_session")

    # -------------------------
    # STEP 1: Preprocess images
    # -------------------------
    prepared_paths = [prepare_image(p) for p in image_paths]

    if multiview and len(prepared_paths) > 1:
        # --- Multi-view path ---
        # /preprocess_images takes a Gallery: List[Dict(image, caption)]
        # Use all images up to MAX_MULTIVIEW_IMAGES
        view_paths = prepared_paths[:MAX_MULTIVIEW_IMAGES]

        print(f"Preprocessing {len(view_paths)} views...")
        gallery_input = [
            {"image": handle_file(p), "caption": None}
            for p in view_paths
        ]
        preprocessed_gallery = client.predict(
            images=gallery_input,
            api_name="/preprocess_images"
        )
        print(f"Preprocessed gallery: {preprocessed_gallery}")
        # preprocessed_gallery is List[Dict(image: filepath, caption: str|None)]

        # The first image in the gallery is used as the primary `image` param;
        # all images (including first) go into `multiimages`
        primary_image = handle_file(preprocessed_gallery[0]["image"])
        multiimages = [
            {"image": handle_file(item["image"]), "caption": item.get("caption")}
            for item in preprocessed_gallery
        ]

        multiimage_algo = "multidiffusion"  # better for true multi-view consistency

    else:
        # --- Single image path ---
        print("Preprocessing single image...")
        preprocessed = client.predict(
            image=handle_file(prepared_paths[0]),
            api_name="/preprocess_image"
        )
        print(f"Preprocessed image: {preprocessed}")

        primary_image = handle_file(preprocessed)
        multiimages = []
        multiimage_algo = "stochastic"

    # -------------------------
    # STEP 2: Get seed
    # -------------------------
    seed = client.predict(
        randomize_seed=True,
        seed=0,
        api_name="/get_seed"
    )
    print(f"Using seed: {seed}")

    # -------------------------
    # STEP 3: image_to_3d
    # -------------------------
    print("Submitting 3D generation job...")
    job = client.submit(
        image=primary_image,
        multiimages=multiimages,
        seed=seed,
        ss_guidance_strength=7.5,
        ss_sampling_steps=12,
        slat_guidance_strength=3.0,
        slat_sampling_steps=12,
        multiimage_algo=multiimage_algo,
        api_name="/image_to_3d"
    )

    res = job.result(timeout=TIMEOUT)
    print(f"3D generation finished. Result: {res}")
    # res is Dict(video: filepath, subtitles: filepath | None)

    # -------------------------
    # STEP 4: Extract GLB
    # -------------------------
    print("Extracting GLB...")
    glb_res = client.submit(
        mesh_simplify=0.95,
        texture_size=1024,
        api_name="/extract_glb"
    ).result(timeout=TIMEOUT)

    print(f"GLB result: {glb_res}")
    # (extracted_glbgaussian: filepath, download_glb: filepath)
    glb_path = glb_res[1]
    target_glb = os.path.join(OUTPUT_DIR, "extracted_mesh.glb")
    shutil.copy(glb_path, target_glb)
    print(f"GLB saved to {target_glb}")

    # -------------------------
    # STEP 5: Extract Gaussians
    # -------------------------
    print("Extracting Gaussians...")
    gs_res = client.submit(
        api_name="/extract_gaussian"
    ).result(timeout=TIMEOUT)

    print(f"Gaussian result: {gs_res}")
    # (extracted_glbgaussian: filepath, download_gaussian: filepath)
    gs_path = gs_res[1]
    target_gs = os.path.join(OUTPUT_DIR, "extracted_gaussians.ply")
    shutil.copy(gs_path, target_gs)
    print(f"Gaussians saved to {target_gs}")


# -------------------------
# Run
# -------------------------
if __name__ == "__main__":
    # Set multiview=False to force single-image mode
    run_trellis_pipeline(SAMPLE_DIR, multiview=True)